# Notebook to Create Torchscript Model from Raw Weights using superdopnet Class

In [ ]:
import sys
import numpy as np
import os

In [2]:
%load_ext autoreload
%autoreload 2

In [3]:
import torch
torch.__version__

'2.5.1'

# Download Model Weights

In [4]:
import requests
import tqdm


def download_checkpoint(filename, url='https://share.hereon.de/index.php/s/kgsExSCm5Pck8Sn/download/superdropnet_final.ckpt'):
    """
    Downloads checkpoint from a URL with a progress bar.

    Args:
        url: The URL of the file to download.
        filename: The name of the file to save the downloaded content to.
    """
    
    response = requests.get(url, stream=True)
    response.raise_for_status()  # Raise an exception for bad status codes

    total_size = int(response.headers.get("content-length", 0))
    block_size = 1024  # 1KB

    print("Downloading checkpoint", url)

    progress_bar = tqdm.tqdm(total=total_size, unit="iB", unit_scale=True)

    with open(filename, "wb") as file:
        for data in response.iter_content(block_size):
            progress_bar.update(len(data))
            file.write(data)

    progress_bar.close()

    if total_size != 0 and progress_bar.n != total_size:
        print("ERROR, something went wrong")


In [ ]:
download_checkpoint('superdropnet_final.ckpt')

100%|██████████| 1.01M/1.01M [00:00<00:00, 9.35MiB/s]


In [ ]:
ckpt = torch.load('superdropnet_final.ckpt', map_location=torch.device('cpu'))
for key, val in ckpt['state_dict'].items():
    print(key, val.shape)
print()

model.layers.0.weight torch.Size([200, 9])
model.layers.0.bias torch.Size([200])
model.layers.1.weight torch.Size([200, 200])
model.layers.1.bias torch.Size([200])
model.layers.2.weight torch.Size([200, 200])
model.layers.2.bias torch.Size([200])
model.layers.3.weight torch.Size([4, 200])
model.layers.3.bias torch.Size([4])



/tmp/ipykernel_88456/3261250732.py:1: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load('../trained_models/superdropnet/superdropnet_final.ckpt', map_location=

In [6]:
random_input = torch.from_numpy(np.random.rand(32, 1, 4)).to(torch.float32)

# Import WrappedSuperdropNet Class that Enforces Physical Constraints

In [1]:
timestep=60

In [8]:
from superdropnet import WrappedSuperdropNet

In [ ]:
wrappedNet = WrappedSuperdropNet(pretrained_path='superdropnet_final.ckpt', timestep=timestep)

/work/ka1176/paul/deep-subgrid/notebooks/../src/cloud_microphysics/superdropnet/superdropnet.py:148: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  saved_ckpt = torch.load(se

In [10]:
# tmp
test_moments = torch.tensor([
                            [0, 1e-11, 1e-3, 1e3], # 0
                             [0, 1e-11, 1e-5, 1e3], # 0
                             [1e-3, 1e3, 1e-3, 1e3], # 1
                             [1e-3, 1e3, 1e-5, 1e3], # 1
                             [1e-3, 1e4, 0, 0], # 1
                             [1e-5, 1e5, 0, 0], 
                             [1e-11, 0, 1e-3, 1e3],
                             [2.7591e-05, 1.9099e+07, 2.2527e-06, 2.4127e+02],
                            [1.1215e-04, 2.0851e+07, 3.1428e-08, 9.0291e+01],
                            [1e-10,1e1,0,1e5]
                            ]).to(torch.float32).unsqueeze(1) # 0
#test_moments = torch.tensor([[[0, 1e-11, 1e-3, 1e3]]]).to(torch.float32)
results = wrappedNet(test_moments)
print(results)

tensor([[[0.0000e+00, 1.0000e-11, 1.0000e-03, 9.6336e+02]],

        [[0.0000e+00, 1.0000e-11, 1.0000e-05, 1.1199e+03]],

        [[1.0000e-03, 0.0000e+00, 1.0000e-03, 0.0000e+00]],

        [[1.0000e-03, 0.0000e+00, 1.0000e-05, 0.0000e+00]],

        [[1.0000e-03, 0.0000e+00, 0.0000e+00, 0.0000e+00]],

        [[4.1914e-06, 1.0000e+05, 5.8086e-06, 0.0000e+00]],

        [[1.0000e-11, 0.0000e+00, 1.0000e-03, 0.0000e+00]],

        [[2.2731e-05, 1.8761e+07, 7.1125e-06, 0.0000e+00]],

        [[1.0739e-04, 2.0851e+07, 4.7947e-06, 0.0000e+00]],

        [[1.0000e-10, 1.0000e+01, 0.0000e+00, 1.0000e+05]]])


In [11]:
print(results-test_moments)

tensor([[[ 0.0000e+00,  0.0000e+00,  0.0000e+00, -1.2213e+01]],

        [[ 0.0000e+00,  0.0000e+00, -9.0949e-13,  3.9958e+01]],

        [[-8.9162e-04, -1.0000e+03,  8.9162e-04, -1.0000e+03]],

        [[-9.4215e-04, -1.0000e+03,  9.4215e-04, -1.0000e+03]],

        [[-6.2166e-04, -1.0000e+04,  6.2166e-04,  0.0000e+00]],

        [[-2.6720e-07,  0.0000e+00,  2.6720e-07,  0.0000e+00]],

        [[-9.9994e-12,  0.0000e+00,  0.0000e+00, -1.0000e+03]],

        [[ 6.6296e-07, -1.1262e+05, -6.6296e-07, -9.4345e+01]],

        [[ 3.1418e-08,  0.0000e+00, -3.1428e-08, -9.0291e+01]],

        [[ 0.0000e+00,  0.0000e+00,  0.0000e+00,  0.0000e+00]]])


# Time Model Speed

This timining is not totally consistent. You get differences of 100us for the same model, which is like 20% of the total time. Be careful when benchmarking!

In [12]:
%%timeit
random_output = wrappedNet(random_input)

713 μs ± 26.4 μs per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


In [15]:
import timeit

# warm up once if needed
_ = wrappedNet(random_input)

loops = 20000

# measure 200 runs with the same inputs
elapsed = timeit.timeit(lambda: wrappedNet(random_input), number=loops)
print(f"avg per run: {elapsed / loops:.6f} s")

avg per run: 0.000578 s


# Save as TorchScript

In [13]:
scripted_model = torch.jit.script(wrappedNet)

In [ ]:
print(scripted_model.code)
filename = 'superdropnet_final_sharma2025_60s.pt'
if os.path.exists(filename):
    print('Will not overwrite: ', filename)
else:
    scripted_model.save(filename)

def forward(self,
    all_moments_in: Tensor) -> Tensor:
  dim_i, dim_k, dim_m, = torch.size(all_moments_in)
  all_moments_in0 = torch.reshape(all_moments_in, [-1, dim_m])
  inputs = (self).create_input(all_moments_in0, )
  backbone = self.backbone
  predictions_updates = (backbone).test_step(inputs, )
  updates_std = self.updates_std
  _0 = torch.mul(predictions_updates, updates_std)
  updates_mean = self.updates_mean
  updates = torch.add(_0, updates_mean)
  _1 = torch.slice(torch.slice(all_moments_in0), 1, None, 4)
  timestep = self.timestep
  moments_out = torch.add(_1, torch.mul(updates, timestep))
  moments_out0 = (self).check_preds(moments_out, all_moments_in0, )
  moments_out1 = torch.reshape(moments_out0, [dim_i, dim_k, dim_m])
  return moments_out1



In [13]:
class IdentityModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
    def forward(self, x):
        return x

In [ ]:
idm = IdentityModel()
sidm = torch.jit.script(idm)
sidm.save('identity_model.pt')

## Try Pruning

In [44]:
wrappedNet

WrappedSuperdropNet(
  (backbone): SuperdropNetInferenceModel(
    (model): plNetwork(
      (layers): ModuleList(
        (0): Linear(in_features=9, out_features=200, bias=True)
        (1-2): 2 x Linear(in_features=200, out_features=200, bias=True)
        (3): Linear(in_features=200, out_features=4, bias=True)
      )
      (dropout): Dropout(p=0.25, inplace=False)
      (activation): ReLU()
    )
  )
)

In [45]:
from itertools import product

In [46]:
Lc = (10**np.linspace(-12., -1, 20)).astype(np.float32)
Lr = (10**np.linspace(-12., -1, 20)).astype(np.float32)
Nc = (10**np.linspace(1, 15, 20)).astype(np.float32)
Nr = (10**np.linspace(1, 15, 20)).astype(np.float32)

inputs = []

for x in product(Lc, Nc, Lr, Nr):
    inputs.append(list(x))

inputs = torch.from_numpy(np.array(inputs)).unsqueeze(1)
inputs.shape

torch.Size([160000, 1, 4])

In [47]:
%%timeit
with torch.no_grad():
    outputs = wrappedNet(inputs)

178 ms ± 890 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


In [48]:
outputs = wrappedNet(inputs).detach().numpy().squeeze()

### Amounts of pruning

In [49]:
from sklearn.metrics import mean_squared_error

In [50]:
from torch.nn.utils import prune
import copy

In [51]:
for amount in [0.01]:#[0.0, 0.01, 0.1, 0.2, 0.5, 0.9]:

    prunedNet = copy.deepcopy(wrappedNet)

    #for i in range(1, 3):
    #    prune.l1_unstructured(prunedNet.backbone.model.layers[i], name="weight", amount=amount)
    #    prune.remove(prunedNet.backbone.model.layers[i], name="weight")

    prunedStrucNet = copy.deepcopy(wrappedNet)
    for i in range(1, 3):
        prune.ln_structured(prunedStrucNet.backbone.model.layers[i], name="weight", amount=amount, n=2, dim=0)
        prune.remove(prunedStrucNet.backbone.model.layers[i], name="weight")

    #pruned_unstruc_outputs = prunedNet(inputs).detach().numpy().squeeze()
    pruned_struc_outputs = prunedStrucNet(inputs).detach().numpy().squeeze()

    print('*'*40)
    print(amount)
    print('*'*40)

    #for i in range(4):
    #    print(mean_squared_error(outputs[:, i], pruned_unstruc_outputs[:, i]))

    for i in range(4):
        print(mean_squared_error(outputs[:, i], pruned_struc_outputs[:, i]))

****************************************
0.01
****************************************
3.162591309712262e-14
141276848.0
3.1631761012590465e-14
205.70887756347656


In [52]:
dim = 1
ix = np.argmax(outputs[:, dim] - pruned_struc_outputs[:, dim])
outputs[ix, dim], pruned_struc_outputs[ix, dim], inputs[ix]


(np.float32(202754720000.0),
 np.float32(202753540000.0),
 tensor([[1.0000e-01, 2.0691e+11, 1.0000e-01, 1.4384e+06]]))

### Half precision

Problem: apparently not supported on CPU

In [53]:
halfNet = copy.deepcopy(wrappedNet).half()
for i in range(4):
    print(halfNet.backbone.model.layers[0].bias.dtype, halfNet.backbone.model.layers[0].weight.dtype, inputs.half().dtype)

torch.float16 torch.float16 torch.float16
torch.float16 torch.float16 torch.float16
torch.float16 torch.float16 torch.float16
torch.float16 torch.float16 torch.float16


In [54]:
half_inputs = inputs.half()
#half_outputs = halfNet(half_inputs)

## Effect of batch size on Speed

In [55]:
extimes = []
batchsizes = []

for n in 2**np.arange(18):
    dt = %timeit -o wrappedNet(inputs[:n])
    extimes.append(dt.average)
    batchsizes.append(n)

318 μs ± 893 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
491 μs ± 964 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
499 μs ± 958 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)
504 μs ± 623 ns per loop (mean ± std. dev. of 7 runs, 1,000 loops each)


KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.loglog(batchsizes, extimes)

In [ ]:
fig, ax = plt.subplots(1, 1)
ax.loglog(batchsizes, np.array(extimes) / np.array(batchsizes), 'o:')
ax.vlines(32, ymin=1e-6, ymax=1e-4)
ax.vlines(64, ymin=1e-6, ymax=1e-4)